In [2]:
import os
import sys
import cv2
import numpy as np
import time
from datetime import datetime
from PyQt5.QtWidgets import (QApplication, QLabel, QWidget, QGridLayout, 
                             QVBoxLayout, QTextEdit, QHBoxLayout, QSizePolicy)
from PyQt5.QtCore import QTimer, Qt
from PyQt5.QtGui import QImage, QPixmap, QFont
from ultralytics import YOLO
from roboflow import Roboflow
from PIL import Image, ImageTk
import threading
def ensure_directory_exists(path):
    if not os.path.exists(path):
        os.makedirs(path)
        print(f"Directorio creado: {path}")
    else:
        print(f"El directorio ya existe: {path}")

In [4]:
# Configuración del entorno
project_root = os.getcwd()  # Detectar dinámicamente el directorio raíz del proyecto
print(f"Directorio raíz del proyecto: {project_root}")

# Verificar y crear directorios necesarios
folders_to_check = [
    os.path.join(project_root, 'Safety-vest---v4-1', 'train'),
    os.path.join(project_root, 'Safety-vest---v4-1', 'valid'),
    os.path.join(project_root, 'Safety-vest---v4-1', 'test')
]

for folder in folders_to_check:
    ensure_directory_exists(folder)

Directorio raíz del proyecto: c:\Users\kenka\Downloads\proyecto1
El directorio ya existe: c:\Users\kenka\Downloads\proyecto1\Safety-vest---v4-1\train
El directorio ya existe: c:\Users\kenka\Downloads\proyecto1\Safety-vest---v4-1\valid
El directorio ya existe: c:\Users\kenka\Downloads\proyecto1\Safety-vest---v4-1\test


In [5]:
# 必要なフォルダーやファイルの存在を確認し、存在しない場合は作成
def ensure_directory_exists(path):
    if not os.path.exists(path):
        os.makedirs(path)
        print(f"フォルダーを作成しました: {path}")
    else:
        print(f"フォルダーは既に存在します: {path}")

# 確認するフォルダー
folders_to_check = [
    os.path.join(project_root, 'Safety-vest---v4-1', 'train'),
    os.path.join(project_root, 'Safety-vest---v4-1', 'valid'),
    os.path.join(project_root, 'Safety-vest---v4-1', 'test')
]

for folder in folders_to_check:
    ensure_directory_exists(folder)

フォルダーは既に存在します: c:\Users\kenka\Downloads\proyecto1\Safety-vest---v4-1\train
フォルダーは既に存在します: c:\Users\kenka\Downloads\proyecto1\Safety-vest---v4-1\valid
フォルダーは既に存在します: c:\Users\kenka\Downloads\proyecto1\Safety-vest---v4-1\test


In [3]:
#!pip install roboflow


In [6]:
import os

ruta_dataset = os.path.join(project_root, 'Safety-vest---v4-1')  # Ruta relativa desde el directorio raíz del proyecto
print(os.listdir(ruta_dataset))

['data.yaml', 'README.dataset.txt', 'README.roboflow.txt', 'test', 'train', 'valid']


In [7]:
# Descarga de datos y modelo desde Roboflow
rf = Roboflow(api_key="HGpn9sxgRvVoq6BS6pDB")
workspace = rf.workspace("prototipo-rro16")
project = workspace.project("safety-vest---v4-he0au")
version = project.version(1)
dataset = version.download("yolov8")

print(f"El dataset y el modelo se han descargado en: {dataset.location}")

loading Roboflow workspace...
loading Roboflow project...
loading Roboflow project...
El dataset y el modelo se han descargado en: c:\Users\kenka\Downloads\proyecto1\Safety-vest---v4-1
El dataset y el modelo se han descargado en: c:\Users\kenka\Downloads\proyecto1\Safety-vest---v4-1


In [8]:
import os

# Especificar la carpeta del conjunto de datos con una ruta relativa desde el directorio raíz del proyecto
dataset_folder = os.path.join(project_root, 'Safety-vest---v4-1')

# Construir las rutas correctas para las carpetas de entrenamiento y validación:
train_folder = os.path.join(dataset_folder, 'train')
valid_folder = os.path.join(dataset_folder, 'valid')

# Listar los archivos en los directorios correctos:
print("Archivos en train:", os.listdir(train_folder))
print("Archivos en valid:", os.listdir(valid_folder))

Archivos en train: ['images', 'labels']
Archivos en valid: ['images', 'labels']


In [9]:
import os

# Especificar la carpeta del conjunto de datos con una ruta relativa desde el directorio raíz del proyecto
dataset_folder = os.path.join(project_root, 'Safety-vest---v4-1')

# Construir la ruta correcta al archivo data.yaml:
data_yaml_path = os.path.join(dataset_folder, 'data.yaml')

# Abrir el archivo:
with open(data_yaml_path, "r") as file:
    print(file.read())

train: ../train/images
val: ../valid/images
test: ../test/images

nc: 4
names: ['helmet', 'not_helmet', 'not_reflective', 'reflective']

roboflow:
  workspace: prototipo-rro16
  project: safety-vest---v4-he0au
  version: 1
  license: CC BY 4.0
  url: https://universe.roboflow.com/prototipo-rro16/safety-vest---v4-he0au/dataset/1


In [10]:
# Entrenamiento del modelo
model = YOLO("yolov8n.pt")
data_yaml_path = os.path.join(project_root, 'Safety-vest---v4-1', 'data.yaml')

try:
    results = model.train(data=data_yaml_path, epochs=20, imgsz=416)
    print("Entrenamiento completado exitosamente.")
except Exception as e:
    print(f"Error durante el entrenamiento: {e}")

Ultralytics 8.3.200  Python-3.12.3 torch-2.8.0+cpu CPU (Intel Core i3-6100T 3.20GHz)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c:\Users\kenka\Downloads\proyecto1\Safety-vest---v4-1\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train11, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=10

In [11]:
# Predicción con una imagen de ejemplo
model_path = os.path.join(project_root, "runs", "detect", "train6", "weights", "best.pt")
if not os.path.exists(model_path):
    raise FileNotFoundError(f"El archivo del modelo no se encontró en la ruta: {model_path}")

model = YOLO(model_path)
image_path = os.path.join(project_root, "istockphoto-1399337320-612x612.jpg")
results = model.predict(image_path, save=True)
print("Predicción completada y guardada.")


image 1/1 c:\Users\kenka\Downloads\proyecto1\istockphoto-1399337320-612x612.jpg: 288x416 2 helmets, 2 reflectives, 101.4ms
Speed: 2.2ms preprocess, 101.4ms inference, 0.9ms postprocess per image at shape (1, 3, 288, 416)
Results saved to C:\Users\kenka\Downloads\proyecto1\runs\detect\predict5
Predicción completada y guardada.
image 1/1 c:\Users\kenka\Downloads\proyecto1\istockphoto-1399337320-612x612.jpg: 288x416 2 helmets, 2 reflectives, 101.4ms
Speed: 2.2ms preprocess, 101.4ms inference, 0.9ms postprocess per image at shape (1, 3, 288, 416)
Results saved to C:\Users\kenka\Downloads\proyecto1\runs\detect\predict5
Predicción completada y guardada.


In [12]:
#CARGA SOLO ESTO PARA QUE EL MODELO CARGUE NUEVAS IMAGENES SIN NECESIDAD DE ENTRENAR EL MODELO DE NUEVO

from ultralytics import YOLO

# 動的に生成されたパスを使用してモデルをロード
model = YOLO(os.path.join(project_root, "runs", "detect", "train6", "weights", "best.pt"))
results = model.predict(os.path.join(project_root, "reactiva-la-construccion.png"), save=True)


image 1/1 c:\Users\kenka\Downloads\proyecto1\reactiva-la-construccion.png: 288x416 3 helmets, 2 reflectives, 77.9ms
Speed: 1.7ms preprocess, 77.9ms inference, 1.2ms postprocess per image at shape (1, 3, 288, 416)
Results saved to C:\Users\kenka\Downloads\proyecto1\runs\detect\predict6
image 1/1 c:\Users\kenka\Downloads\proyecto1\reactiva-la-construccion.png: 288x416 3 helmets, 2 reflectives, 77.9ms
Speed: 1.7ms preprocess, 77.9ms inference, 1.2ms postprocess per image at shape (1, 3, 288, 416)
Results saved to C:\Users\kenka\Downloads\proyecto1\runs\detect\predict6


APLICACION

In [13]:
import cv2
import threading
import tkinter as tk
from tkinter import ttk
from PIL import Image, ImageTk
from ultralytics import YOLO
from datetime import datetime
import os

# Construir la ruta al modelo dinámicamente
model_path = os.path.join(project_root, "runs", "detect", "train6", "weights", "best.pt")

# Verificar si el archivo del modelo existe
if not os.path.exists(model_path):
    raise FileNotFoundError(f"El archivo del modelo no se encontró en la ruta: {model_path}")

# Carga el modelo entrenado
model = YOLO(model_path)

# Rutas de los videos
videos = [
    os.path.join(project_root, "CONSTRUCCIONES BUENVIVIR - VIDEO CORPORATIVO.mp4"),
    os.path.join(project_root, "Curso de Seguridad en la construcción.mp4"),
    os.path.join(project_root, "VIDEO DE SEGURIDAD.mp4"),
    os.path.join(project_root, "videoplayback.mp4")
]

# Contador de alertas
alert_count = {f"CAM {i+1}": 0 for i in range(len(videos))}

# Configurar ventana principal
root = tk.Tk()
root.title("🛡️ Centro de Monitoreo - CCTV Seguridad")
root.geometry("1280x800")

# Dividir ventana
main_frame = tk.PanedWindow(root, orient=tk.VERTICAL)
main_frame.pack(fill=tk.BOTH, expand=True)

# Frame para cámaras
cams_frame = tk.Frame(main_frame, bg="gray")
main_frame.add(cams_frame, stretch='always')

# Frame para historial y ranking
bottom_frame = tk.Frame(main_frame, bg="black", height=200)
main_frame.add(bottom_frame)

# Subdividir historial y ranking
history_frame = tk.Frame(bottom_frame, bg="black")
ranking_frame = tk.Frame(bottom_frame, bg="black", width=200)
history_frame.pack(side='left', fill='both', expand=True)
ranking_frame.pack(side='right', fill='y')

# Configurar cámaras en 2x2
labels = []
name_labels = []
for i in range(4):
    frame = tk.Frame(cams_frame, bd=2, relief=tk.RIDGE)
    frame.grid(row=i//2, column=i%2, padx=5, pady=5, sticky="nsew")

    name = tk.Label(frame, text=f"CAM {i+1}", bg="black", fg="white", font=("Arial", 14, "bold"))
    name.pack(fill='x')
    name_labels.append(name)

    label = tk.Label(frame)
    label.pack(expand=True, fill='both')
    labels.append(label)

# Que las celdas se expandan correctamente
for i in range(2):
    cams_frame.grid_rowconfigure(i, weight=1)
    cams_frame.grid_columnconfigure(i, weight=1)

# Historial
history_label = tk.Label(history_frame, text="📋 Historial de Alertas:", anchor="w", fg="lime", bg="black", font=("Arial", 12, "bold"))
history_label.pack(fill='x')

history_text = tk.Text(history_frame, bg="black", fg="lime", font=("Consolas", 10), height=10)
history_text.pack(side='left', fill='both', expand=True)

scrollbar = ttk.Scrollbar(history_frame, command=history_text.yview)
scrollbar.pack(side='right', fill='y')
history_text.config(yscrollcommand=scrollbar.set, state='disabled')

# Ranking de alertas
ranking_label = tk.Label(ranking_frame, text="🏆 Ranking:", anchor="center", fg="orange", bg="black", font=("Arial", 12, "bold"))
ranking_label.pack(fill='x')
ranking_text = tk.Label(ranking_frame, justify='left', bg="black", fg="orange", font=("Consolas", 10))
ranking_text.pack(fill='both', expand=True)

# Función para registrar alertas
def log_alert(camera):
    alert_count[camera] += 1
    now = datetime.now().strftime("[%Y-%m-%d %H:%M:%S]")
    history_text.config(state='normal')
    history_text.insert('end', f"⚠️ {now} {camera}: Falta de elementos de seguridad detectada.\n")
    history_text.see('end')
    history_text.config(state='disabled')
    update_ranking()

# Actualizar ranking
def update_ranking():
    sorted_cams = sorted(alert_count.items(), key=lambda x: x[1], reverse=True)
    text = ""
    for cam, count in sorted_cams:
        text += f"{cam}: {count} alertas\n"
    ranking_text.config(text=text)

# Procesar cada video
def process_video(idx, video_path):
    cap = cv2.VideoCapture(video_path)
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
            continue

        results = model.predict(source=frame, conf=0.4, imgsz=416, verbose=False)
        for r in results:
            names = r.names
            for c in r.boxes.cls:
                label = names[int(c)]
                if label == "not_reflective":
                    log_alert(f"CAM {idx+1}")

        annotated_frame = results[0].plot()
        rgb_image = cv2.cvtColor(annotated_frame, cv2.COLOR_BGR2RGB)
        img = ImageTk.PhotoImage(Image.fromarray(rgb_image))

        labels[idx].config(image=img)
        labels[idx].image = img

    cap.release()

# Lanzar hilos para cada video
for i, path in enumerate(videos):
    threading.Thread(target=process_video, args=(i, path), daemon=True).start()

root.mainloop()

Ultralytics 8.3.200  Python-3.12.3 torch-2.8.0+cpu CPU (Intel Core i3-6100T 3.20GHz)
Ultralytics 8.3.200  Python-3.12.3 torch-2.8.0+cpu CPU (Intel Core i3-6100T 3.20GHz)
Ultralytics 8.3.200  Python-3.12.3 torch-2.8.0+cpu CPU (Intel Core i3-6100T 3.20GHz)
Ultralytics 8.3.200  Python-3.12.3 torch-2.8.0+cpu CPU (Intel Core i3-6100T 3.20GHz)
Ultralytics 8.3.200  Python-3.12.3 torch-2.8.0+cpu CPU (Intel Core i3-6100T 3.20GHz)
Model summary (fused): 72 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs
Model summary (fused): 72 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs
Model summary (fused): 72 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs
Model summary (fused): 72 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs
Model summary (fused): 72 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs
Model summary (fused): 72 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs


Exception in thread Thread-21 (process_video):
Traceback (most recent call last):
  File "c:\Users\kenka\AppData\Local\Programs\Python\Python312\Lib\threading.py", line 1073, in _bootstrap_inner
    self.run()
  File "C:\Users\kenka\AppData\Roaming\Python\Python312\site-packages\ipykernel\ipkernel.py", line 772, in run_closure
    _threading_Thread_run(self)
  File "c:\Users\kenka\AppData\Local\Programs\Python\Python312\Lib\threading.py", line 1010, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\kenka\AppData\Local\Temp\ipykernel_5716\1525129575.py", line 128, in process_video
  File "c:\Users\kenka\AppData\Local\Programs\Python\Python312\Lib\site-packages\PIL\ImageTk.py", line 129, in __init__
    self.__photo = tkinter.PhotoImage(**kw)
                   ^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\kenka\AppData\Local\Programs\Python\Python312\Lib\tkinter\__init__.py", line 4151, in __init__
    Image.__init__(self, 'photo', name, cnf, master, **kw)
  File "c:\Use

In [ ]:
import cv2
from ultralytics import YOLO
from datetime import datetime
import os

# Definir el directorio raíz del proyecto
project_root = os.getcwd()  # Detectar dinámicamente el directorio raíz del proyecto

# Construir la ruta al modelo dinámicamente
model_path = os.path.join(project_root, "runs", "detect", "train6", "weights", "best.pt")

# Verificar si el archivo del modelo existe
if not os.path.exists(model_path):
    raise FileNotFoundError(f"El archivo del modelo no se encontró en la ruta: {model_path}")

# Carga el modelo entrenado
model = YOLO(model_path)

# Intenta abrir la cámara principal
cap = cv2.VideoCapture(0)

# Verifica si se abrió correctamente
if not cap.isOpened():
    print("No se pudo acceder a la cámara.")
    exit()

while True:
    ret, frame = cap.read()
    if not ret:
        print("No se recibió frame desde la cámara.")
        break

    # Predicción YOLOv8
    results = model.predict(source=frame, save=False, imgsz=416, conf=0.4, verbose=False)
    annotated_frame = results[0].plot()  # Dibujar resultados sobre el frame

    # Fecha y hora
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    cv2.putText(annotated_frame, now, (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    # Mostrar ventana
    cv2.imshow("Vista de la Cámara", annotated_frame)

    # Salir con ESC
    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

No se pudo acceder a la cámara.
No se recibió frame desde la cámara.


error: OpenCV(4.10.0) D:\a\opencv-python\opencv-python\opencv\modules\highgui\src\window.cpp:1295: error: (-2:Unspecified error) The function is not implemented. Rebuild the library with Windows, GTK+ 2.x or Cocoa support. If you are on Ubuntu or Debian, install libgtk2.0-dev and pkg-config, then re-run cmake or configure script in function 'cvDestroyAllWindows'


: 

In [6]:
import sys
import os
import cv2
import numpy as np
import time
from datetime import datetime
from PyQt5.QtWidgets import (QApplication, QLabel, QWidget, QGridLayout, 
                             QVBoxLayout, QTextEdit, QHBoxLayout, QSizePolicy)
from PyQt5.QtCore import QTimer, Qt
from PyQt5.QtGui import QImage, QPixmap, QFont
from ultralytics import YOLO

class CameraWidget(QWidget):
    def __init__(self, title, camera_index=None, model=None, log_widget=None, ranking_counter=None):
        super().__init__()
        self.title = title
        self.camera_index = camera_index
        self.model = model
        self.log_widget = log_widget
        self.ranking_counter = ranking_counter
        self.ultima_alerta = 0

        self.cap = cv2.VideoCapture(camera_index) if camera_index is not None else None

        self.label = QLabel()
        self.label.setStyleSheet("background-color: black; color: white;")
        self.label.setAlignment(Qt.AlignCenter)
        self.label.setSizePolicy(QSizePolicy.Expanding, QSizePolicy.Expanding)
        self.label.setScaledContents(True)

        self.titleLabel = QLabel(self.title)
        self.titleLabel.setAlignment(Qt.AlignCenter)
        self.titleLabel.setFont(QFont("Arial", 12, QFont.Bold))
        self.titleLabel.setStyleSheet("color: white;")

        self.alertaTimerLabel = QLabel()
        self.alertaTimerLabel.setAlignment(Qt.AlignCenter)
        self.alertaTimerLabel.setStyleSheet("color: #FFA500; font-size: 10pt;")

        layout = QVBoxLayout()
        layout.addWidget(self.titleLabel)
        layout.addWidget(self.label)
        layout.addWidget(self.alertaTimerLabel)
        self.setLayout(layout)

        if self.cap and self.cap.isOpened():
            self.timer = QTimer()
            self.timer.timeout.connect(self.update_frame)
            self.timer.start(30)
        else:
            self.show_no_signal()

    def update_frame(self):
        ret, frame = self.cap.read()
        if not ret:
            self.show_no_signal()
            return

        results = self.model.predict(source=frame, imgsz=416, conf=0.4, verbose=False)
        annotated_frame = results[0].plot()

        clases_detectadas = [results[0].names[int(cls)] for cls in results[0].boxes.cls]

        casco = 'helmet' in clases_detectadas
        chaleco = 'reflective' in clases_detectadas

        if not (casco and chaleco):
            ahora = time.time()
            if ahora - self.ultima_alerta >= 10:
                self.ultima_alerta = ahora
                timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
                alerta = f"[{timestamp}] {self.title}: Falta de elementos de seguridad detectada."
                if self.log_widget:
                    self.log_widget.append(f"<span style='color: lime;'>{alerta}</span>")
                if self.ranking_counter is not None:
                    self.ranking_counter[self.title] += 1

        # Mostrar tiempo restante para nueva alerta
        if self.ultima_alerta != 0:
            tiempo_restante = max(0, 10 - int(time.time() - self.ultima_alerta))
            minutos = tiempo_restante // 60
            segundos = tiempo_restante % 60
            self.alertaTimerLabel.setText(f"Próxima alerta en: {minutos:02d}:{segundos:02d}")
        else:
            self.alertaTimerLabel.setText("")

        timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        cv2.putText(annotated_frame, timestamp, (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)

        rgb_image = cv2.cvtColor(annotated_frame, cv2.COLOR_BGR2RGB)
        h, w, ch = rgb_image.shape
        bytes_per_line = ch * w
        qt_image = QImage(rgb_image.data, w, h, bytes_per_line, QImage.Format_RGB888)
        self.label.setPixmap(QPixmap.fromImage(qt_image))

    def show_no_signal(self):
        self.label.setText("SIN SEÑAL")
        self.label.setStyleSheet("background-color: #222; color: #FF4444; font-size: 22px;")

class MainWindow(QWidget):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Sistema de Vigilancia")
        self.setStyleSheet("background-color: #333; color: white;")
        self.layout = QGridLayout()

        self.ranking_counter = {
            "Cam 1 - Área de Acceso y Control": 0,
            "Cam 2 - Área de Almacenamiento": 0,
            "Cam 3 - Área de mezcla de concreto": 0,
            "Cam 4 - Área de armado de acero": 0
        }

        self.alert_log = QTextEdit()
        self.alert_log.setReadOnly(True)
        self.alert_log.setStyleSheet("background-color: black; color: lime; font-family: Consolas;")

        self.ranking_label = QLabel()
        self.ranking_label.setStyleSheet("color: orange; font-size: 10pt; font-weight: bold;")
        self.update_ranking()

        # Construir la ruta al modelo dinámicamente
        model_path = os.path.join(project_root, "runs", "detect", "train6", "weights", "best.pt")

        # Verificar si el archivo del modelo existe
        if not os.path.exists(model_path):
            raise FileNotFoundError(f"El archivo del modelo no se encontró en la ruta: {model_path}")

        self.model = YOLO(model_path)

        cameras = [
            ("Cam 1 - Área de Acceso y Control", 0),
            ("Cam 2 - Área de Almacenamiento", None),
            ("Cam 3 - Área de mezcla de concreto", None),
            ("Cam 4 - Área de armado de acero", None)
        ]

        positions = [(0, 0), (0, 1), (1, 0), (1, 1)]
        for pos, (name, index) in zip(positions, cameras):
            cam_widget = CameraWidget(name, index, self.model, self.alert_log, self.ranking_counter)
            self.layout.addWidget(cam_widget, *pos)

        right_panel = QVBoxLayout()
        right_panel.addWidget(QLabel("\u26A1 Historial de Alertas:"))
        right_panel.addWidget(self.alert_log)
        right_panel.addWidget(QLabel("\u2696 Ranking:"))
        right_panel.addWidget(self.ranking_label)

        main_layout = QHBoxLayout()
        main_layout.addLayout(self.layout, 3)
        main_layout.addLayout(right_panel, 1)

        self.setLayout(main_layout)

        self.timer_ranking = QTimer()
        self.timer_ranking.timeout.connect(self.update_ranking)
        self.timer_ranking.start(10000)

    def update_ranking(self):
        sorted_ranking = sorted(self.ranking_counter.items(), key=lambda x: x[1], reverse=True)
        ranking_text = "<br>".join([f"{name}: {count} alertas" for name, count in sorted_ranking])
        self.ranking_label.setText(ranking_text)

if __name__ == "__main__":
    app = QApplication(sys.argv)
    window = MainWindow()
    window.showMaximized()
    sys.exit(app.exec_())

SystemExit: 0

To exit: use 'exit', 'quit', or Ctrl-D.
